# Sobel Derivative

Compute images derivatives in both x and y directions

The Sobel kernel is:
$$g_x = 1/8
\begin{bmatrix}
1 & 0 & -1 \\
2 & 0 & -2 \\
1 & 0 & -1
\end{bmatrix}
$$

$$g_y = 1/8
\begin{bmatrix}
1 & 2 & 1 \\
0 & 0 & 0 \\
-1 & -2 & -1
\end{bmatrix}
$$


In [ ]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [26]:
import numpy as np
from PIL import Image
from src.utils import round_matrix

### Compute Derivative function

In [6]:
def sobel_derivative(image, image_width, image_height, direction='x'):
    # 3x3 Sobel kernels
    g_x = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]) / 8
    g_y = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]]) / 8

    kernel_size = len(g_x)

    extended_image = np.pad(image, pad_width=1, mode="edge")

    # # debug: print the extended image
    # print("Extended image:")
    # print(extended_image)

    filtered_image = np.zeros_like(image, dtype=np.float64)
    for i in range(image_height):
        for j in range(image_width):
            region = extended_image[i : i + kernel_size, j : j + kernel_size]
            if direction == 'x':
                filtered_image[i, j] = np.sum(region * g_x)
            else:
                filtered_image[i, j] = np.sum(region * g_y)
    return filtered_image
    
    
def compute_image_derivatives(image, image_width, image_height):
    I_x = sobel_derivative(image, image_width, image_height, direction='x')
    I_y = sobel_derivative(image, image_width, image_height, direction='y')
    return I_x**2, I_y**2, I_x * I_y

#### Testing

Given matrix

$$ I = 
\begin{bmatrix}
49 & 223 & 46 & 174 & 128 \\
84 & 165 & 223 & 233 & 97 \\
168 & 87 & 187 & 42 & 62 \\
115 & 128 & 100 & 202 & 3 \\
71 & 89 & 76 & 50 & 244
\end{bmatrix}
$$

Then the extending matrix using repeated at the border (BorderBoundaryPadding). This means that at the four edges of the image, as well as at the four corner locations, the pixel intensities are extended (copied) to the outside of the image.

$$ I_{extended} = 
\begin{bmatrix}
\underline{49} & \underline{49} & \underline{223} & \underline{46} & \underline{174} & \underline{128} & \underline{128} \\
\underline{49} & 49 & 223 & 46 & 174 & 128 & \underline{128} \\
\underline{84} & 84 & 165 & 223 & 233 & 97 & \underline{97} \\
\underline{168} & 168 & 87 & 187 & 42 & 62 & \underline{62} \\
\underline{115} & 115 & 128 & 100 & 202 & 3 & \underline{3} \\
\underline{71} & \textbf{71} & 89 & 76 & 50 & 244 & \underline{244} \\
\underline{71} & \underline{71} & \underline{89} & \underline{76} & \underline{50} & \underline{244} & \underline{244} 
\end{bmatrix}
$$

Calculate the value at bottom left, aka **71**, of:

$$
g_x[0][4] = (115 + 71*(1+2) - 128 - 89*(1+ 2))/8=-67/8=-8.375
$$

$$
g_y[0][4] = (115 * (1+2) + 128 -71 * (1+2) - 89)/8=-171/8=21.375
$$

In [7]:
I = np.array([
    [49 , 223 , 46 , 174 , 128],
    [84 , 165 , 223 , 233 , 97],
    [168 , 87 , 187 , 42 , 62],
    [115 , 128 , 100 , 202 , 3],
    [71 , 89 , 76 , 50 , 244]
])

I_x = sobel_derivative(I, len(I[0]), len(I), direction='x')
I_y = sobel_derivative(I, len(I[0]), len(I), direction='y')

print("I_x:")
print(I_x)
print("I_y:")
print(I_y)

I_x:
[[-75.375 -16.25    9.875 -15.     34.25 ]
 [-31.875 -36.75   -5.25   36.875  37.25 ]
 [  8.5   -20.25   -6.5    59.125  36.875]
 [  4.625   0.75   -8.     18.875  23.   ]
 [ -8.375   0.      5.375 -50.875 -47.875]]
I_y:
[[ -5.875 -12.    -44.375 -33.      4.25 ]
 [-27.625   1.5    -1.75   23.625  41.25 ]
 [ -7.     20.75   39.25   34.875  39.125]
 [ 36.125  25.5    26.5   -10.875 -69.25 ]
 [ 21.375  18.25   29.875  10.875 -71.375]]


In [28]:
print(round_matrix(I_x))
print(round_matrix(I_y))
print(round_matrix(I_x)**2)
print(round_matrix(I_y)**2)
print(round_matrix(I_x * I_y))

[[-75 -16  10 -15  34]
 [-32 -37  -5  37  37]
 [  9 -20  -7  59  37]
 [  5   1  -8  19  23]
 [ -8   0   5 -51 -48]]
[[ -6 -12 -44 -33   4]
 [-28   2  -2  24  41]
 [ -7  21  39  35  39]
 [ 36  26  27 -11 -69]
 [ 21  18  30  11 -71]]
[[5625  256  100  225 1156]
 [1024 1369   25 1369 1369]
 [  81  400   49 3481 1369]
 [  25    1   64  361  529]
 [  64    0   25 2601 2304]]
[[  36  144 1936 1089   16]
 [ 784    4    4  576 1681]
 [  49  441 1521 1225 1521]
 [1296  676  729  121 4761]
 [ 441  324  900  121 5041]]
[[  443   195  -438   495   146]
 [  881   -55     9   871  1537]
 [  -60  -420  -255  2062  1443]
 [  167    19  -212  -205 -1593]
 [ -179     0   161  -553  3417]]


### Calculate Image derivative

In [ ]:
left_image = np.array(Image.open("assets/gaussian_filter_left_image.png").convert('L'))
right_image = np.array(Image.open("assets/gaussian_filter_right_image.png").convert('L'))

left_Ix2, left_Iy2, left_IxIy = compute_image_derivatives(left_image, len(left_image[0]), len(left_image))
right_Ix2, right_Iy2, right_IxIy = compute_image_derivatives(right_image, len(right_image[0]), len(right_image))

In [16]:
print("Right image I_x^2:")
print(right_Ix2)
print("Right image I_y^2:")
print(right_Iy2)
print("Right image I_xI_y:")
print(right_IxIy)

Right image I_x^2:
[[3.0625000e+00 9.7656250e+00 3.0625000e+00 ... 4.5156250e+00
  2.3765625e+01 6.2500000e+00]
 [1.0000000e+00 3.5156250e+00 1.0000000e+00 ... 5.0625000e+00
  1.8062500e+01 5.0625000e+00]
 [1.5625000e-02 1.4062500e-01 6.2500000e-02 ... 1.8906250e+00
  6.2500000e+00 1.8906250e+00]
 ...
 [1.4062500e-01 1.5625000e-02 6.2500000e+00 ... 4.7265625e+01
  4.0640625e+01 3.0625000e+00]
 [1.5625000e-02 5.6250000e-01 1.6000000e+01 ... 5.4390625e+01
  3.9062500e+01 3.5156250e+00]
 [1.4062500e-01 1.8906250e+00 1.4062500e+01 ... 3.3062500e+01
  2.5000000e+01 2.6406250e+00]]
Right image I_y^2:
[[1.5625000e+00 3.9062500e-01 0.0000000e+00 ... 4.5156250e+00
  4.5156250e+00 4.0000000e+00]
 [4.0000000e+00 3.9062500e-01 5.6250000e-01 ... 1.2250000e+01
  9.0000000e+00 5.0625000e+00]
 [1.5625000e-02 7.6562500e-01 3.0625000e+00 ... 1.2656250e+00
  1.0000000e+00 8.2656250e+00]
 ...
 [4.3890625e+01 4.0640625e+01 2.0250000e+01 ... 8.2656250e+00
  2.3765625e+01 3.3062500e+01]
 [1.8906250e+00 5.062

In [17]:
print("Right image I_x^2:")
print(right_Ix2)
print("Right image I_y^2:")
print(right_Iy2)
print("Right image I_xI_y:")
print(right_IxIy)

Right image I_x^2:
[[3.0625000e+00 9.7656250e+00 3.0625000e+00 ... 4.5156250e+00
  2.3765625e+01 6.2500000e+00]
 [1.0000000e+00 3.5156250e+00 1.0000000e+00 ... 5.0625000e+00
  1.8062500e+01 5.0625000e+00]
 [1.5625000e-02 1.4062500e-01 6.2500000e-02 ... 1.8906250e+00
  6.2500000e+00 1.8906250e+00]
 ...
 [1.4062500e-01 1.5625000e-02 6.2500000e+00 ... 4.7265625e+01
  4.0640625e+01 3.0625000e+00]
 [1.5625000e-02 5.6250000e-01 1.6000000e+01 ... 5.4390625e+01
  3.9062500e+01 3.5156250e+00]
 [1.4062500e-01 1.8906250e+00 1.4062500e+01 ... 3.3062500e+01
  2.5000000e+01 2.6406250e+00]]
Right image I_y^2:
[[1.5625000e+00 3.9062500e-01 0.0000000e+00 ... 4.5156250e+00
  4.5156250e+00 4.0000000e+00]
 [4.0000000e+00 3.9062500e-01 5.6250000e-01 ... 1.2250000e+01
  9.0000000e+00 5.0625000e+00]
 [1.5625000e-02 7.6562500e-01 3.0625000e+00 ... 1.2656250e+00
  1.0000000e+00 8.2656250e+00]
 ...
 [4.3890625e+01 4.0640625e+01 2.0250000e+01 ... 8.2656250e+00
  2.3765625e+01 3.3062500e+01]
 [1.8906250e+00 5.062

### Verfiy the result by comparing sample filtered images

In [18]:
expected_left_Ix = np.load('assets/sobel_derivative_expected_outputs/step2_left_ix_square.npy')
expected_left_Iy = np.load('assets/sobel_derivative_expected_outputs/step2_left_iy_square.npy')
expected_left_IxIy = np.load('assets/sobel_derivative_expected_outputs/step2_left_ixiy.npy')

expected_right_Ix = np.load('assets/sobel_derivative_expected_outputs/step2_right_ix_square.npy')
expected_right_Iy = np.load('assets/sobel_derivative_expected_outputs/step2_right_iy_square.npy')
expected_right_IxIy = np.load('assets/sobel_derivative_expected_outputs/step2_right_ixiy.npy')

print("Verifying results are identical to sample images:")
print(np.array_equal(expected_left_Ix, left_Ix2))
print(np.array_equal(expected_left_Iy, left_Iy2))
print(np.array_equal(expected_left_IxIy, left_IxIy))
print(np.array_equal(expected_right_Ix, right_Ix2))
print(np.array_equal(expected_right_Iy, right_Iy2))
print(np.array_equal(expected_right_IxIy, right_IxIy))

Verifying results are identical to sample images:
True
True
True
True
True
True
